# Generate paper tables from `results/`

Per CLAUDE.md Section 9 + Day 6 end-check, this notebook reads the per-cell
summaries written by `harness.run_corpus` / `harness.run_benign` and emits:

* **Table 1** — ASR by scenario × orchestrator × CapGuard config (with Wilson 95% CIs).
* **Table 2** — FPR by orchestrator (CapGuard ON, Wilson 95% CIs).
* **Table 3** — Latency overhead per tool call (median, p95).
* **Commit-race** — case-study numbers.

All reported proportions are accompanied by Wilson 95% intervals (`harness.metrics.Proportion.wilson_ci`).

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
REPO_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(REPO_ROOT))
from harness.metrics import Proportion, latency_summary

In [ ]:
RESULTS = REPO_ROOT / 'results'
OUT = REPO_ROOT / 'paper_outputs'
OUT.mkdir(exist_ok=True, parents=True)

def _read_corpus(p):
    p = Path(p)
    if not p.exists():
        return pd.DataFrame()
    return pd.DataFrame([json.loads(l) for l in p.read_text().splitlines() if l.strip()])

baseline = _read_corpus(RESULTS / 'baseline_frontier' / 'corpus_summary.jsonl')
guarded  = _read_corpus(RESULTS / 'capguard_frontier' / 'corpus_summary.jsonl')
ablation = _read_corpus(RESULTS / 'ablation_openmodel' / 'corpus_summary.jsonl')
benign   = _read_corpus(RESULTS / 'benign_frontier' / 'benign_summary.jsonl')
all_cells = pd.concat([baseline, guarded, ablation], ignore_index=True) if not baseline.empty else pd.DataFrame()

## Table 1 — ASR by scenario × orchestrator × CapGuard config

In [ ]:
if all_cells.empty:
    print('No results yet. Run `bash scripts/run_all.sh` first.')
else:
    grouped = (all_cells.groupby(['scenario', 'orchestrator', 'capguard'])
               .agg(successes=('successes', 'sum'), n=('k', 'sum')).reset_index())
    grouped['asr'] = grouped['successes'] / grouped['n']
    grouped['ci'] = grouped.apply(lambda r: Proportion(int(r['successes']), int(r['n'])).wilson_ci(), axis=1)
    grouped[['ci_lo', 'ci_hi']] = pd.DataFrame(grouped['ci'].tolist(), index=grouped.index)
    grouped = grouped.drop(columns=['ci'])
    grouped.to_csv(OUT / 'table1_asr.csv', index=False)
    grouped

## Table 2 — FPR by orchestrator (CapGuard ON)

In [ ]:
if benign.empty:
    print('No benign results yet.')
else:
    fpr = (benign[benign['capguard'] == 'on']
           .groupby('orchestrator')
           .agg(blocked=('blocked_runs', 'sum'), n=('k', 'sum')).reset_index())
    fpr['fpr'] = fpr['blocked'] / fpr['n']
    fpr['ci'] = fpr.apply(lambda r: Proportion(int(r['blocked']), int(r['n'])).wilson_ci(), axis=1)
    fpr[['ci_lo', 'ci_hi']] = pd.DataFrame(fpr['ci'].tolist(), index=fpr.index)
    fpr = fpr.drop(columns=['ci'])
    fpr.to_csv(OUT / 'table2_fpr.csv', index=False)
    fpr

## Table 3 — Latency overhead per tool call (median, p95)

In [ ]:
# Latency is reconstructed from the agent JSONL traces. Each `tool_call`
# is paired with its matching `tool_result` (same tool_call_id) and the
# delta is the per-call wall time. CapGuard ON vs OFF is the comparison
# of interest for Section 6 of the paper.

from collections import defaultdict

def _gather_latencies(root):
    root = Path(root)
    if not root.exists():
        return []
    out = []
    for jl in root.rglob('agent.jsonl'):
        starts = {}
        for line in jl.read_text().splitlines():
            if not line.strip():
                continue
            r = json.loads(line)
            tcid = r.get('tool_call_id')
            if r.get('kind') == 'tool_call' and tcid:
                starts[tcid] = r.get('ts', 0)
            elif r.get('kind') == 'tool_result' and tcid in starts:
                out.append(1000 * (r.get('ts', 0) - starts[tcid]))
                del starts[tcid]
    return out

rows = []
for label, path in [('baseline', RESULTS / 'baseline_frontier'),
                    ('capguard', RESULTS / 'capguard_frontier')]:
    s = latency_summary(_gather_latencies(path))
    s['config'] = label
    rows.append(s)
lat = pd.DataFrame(rows)
lat.to_csv(OUT / 'table3_latency.csv', index=False)
lat

## Commit-race case study

In [ ]:
def _read_summary(p):
    p = Path(p) / 'run_*' / '..' / 'summary.jsonl'
    candidates = list((Path(p).parent.parent).glob('**/summary.jsonl'))
    return [json.loads(l) for c in candidates for l in c.read_text().splitlines() if l.strip()]

cr_off = _read_summary(RESULTS / 'commit_race' / 'baseline')
cr_on  = _read_summary(RESULTS / 'commit_race' / 'capguard')
def _rate(rs):
    if not rs:
        return None
    n = len(rs)
    s = sum(1 for r in rs if r.get('attack_success'))
    p = Proportion(s, n)
    return {'n': n, 'successes': s, 'asr': p.rate, 'ci': p.wilson_ci()}
cr = pd.DataFrame([
    {'config': 'capguard_off', **(_rate(cr_off) or {})},
    {'config': 'capguard_on',  **(_rate(cr_on)  or {})},
])
cr.to_csv(OUT / 'commit_race.csv', index=False)
cr